# Path integration

**`path_integrate`** integrates body-frame increments into global pose; **`LinearModel`** fits a lightweight regression from commanded deltas to observed state (tutorial-friendly substitute for larger 1.x pipelines).

**`RandomExplorationController`** composes `HybridTurningController` with random bouts of walking and turning to collect short trajectories for analysis.


In [ ]:
import numpy as np
from flygym import Simulation
from flygym.anatomy import ContactBodiesPreset
from flygym.utils.math import Rotation3D
from flygym_demo.examples import make_walking_fly, run_closed_loop
from flygym_demo.examples.locomotion import HybridTurningController
from flygym_demo.examples.path_integration import LinearModel, path_integrate, RandomExplorationController
from flygym_demo.examples.worlds.terrain import FlatDemoWorld

deltas = np.array([[1.0, 0.0, 0.05], [0.5, 0.1, -0.02], [0.2, 0.0, 0.0]])
states = path_integrate(np.zeros(3), deltas)
model = LinearModel().fit(deltas, states[1:])
print("predict", model.predict(deltas[:1]))

fly = make_walking_fly(add_camera=False)
world = FlatDemoWorld()
world.add_fly(
    fly,
    spawn_position=[0, 0, 1.5],
    spawn_rotation=Rotation3D("quat", [1, 0, 0, 0]),
    bodysegs_with_ground_contact=ContactBodiesPreset.TIBIA_TARSUS_ONLY,
    add_ground_contact_sensors=False,
)
sim = Simulation(world)
walk = HybridTurningController(sim.timestep)
explorer = RandomExplorationController(walk, bout_duration_s=(0.05, 0.12), seed=1)
records = run_closed_loop(sim, explorer, 0.01, fly_name=fly.name)
print("n_records", len(records))
